# RDD Hands-On — PySpark

Practical companion to `4RDD.md`. Walks through:

1. Setting up Spark
2. Creating RDDs
3. Transformations (lazy) vs Actions (eager)
4. Common operations: `map`, `filter`, `flatMap`, `distinct`
5. Key/value RDDs: `reduceByKey`, `groupByKey`
6. Caching for reuse
7. Lineage demonstration
8. Word count — the canonical RDD example

**Requirements:**
```
pip install pyspark
```

## 1 — Create a SparkSession

Every PySpark program starts here. `SparkSession` gives you the `sparkContext` (sc) used for creating RDDs.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('RDD-handson') \
    .master('local[*]') \
    .getOrCreate()

sc = spark.sparkContext
print('Spark version:', spark.version)
print('Number of partitions on this machine:', sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/16 22:56:48 WARN Utils: Your hostname, Navspeak, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/16 22:56:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 22:56:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Number of partitions on this machine: 16


`local[*]` means run on the local machine using all available cores. For real clusters you'd set this differently (e.g., `yarn`).

Suppress noisy logs (optional):

In [2]:
sc.setLogLevel('WARN')

## 2 — Creating RDDs

Two main ways:
1. **From a Python collection** — `sc.parallelize(list)`
2. **From a file** — `sc.textFile(path)`

In [3]:
# Method 1: parallelize a list
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(data)

print('Type:', type(rdd))
print('Num partitions:', rdd.getNumPartitions())

Type: <class 'pyspark.core.rdd.RDD'>
Num partitions: 16


In [5]:
# You can specify the number of partitions explicitly
rdd_4 = sc.parallelize(data, numSlices=4)
print('Custom partitions:', rdd_4.getNumPartitions())

# See how data is split across partitions
print('Per-partition view:', rdd_4.glom().collect())

Custom partitions: 4
Per-partition view: [[1, 2], [3, 4], [5, 6], [7, 8, 9, 10]]


`glom()` returns one list per partition — useful to see how Spark splits data.

## 3 — Transformations vs Actions

**Most important concept in Spark:**

```
TRANSFORMATIONS:
   - Return a NEW RDD
   - LAZY — don't execute, just build the DAG
   - Examples: map, filter, flatMap, distinct

ACTIONS:
   - Return a VALUE to the driver
   - EAGER — trigger execution of the DAG
   - Examples: count, collect, take, reduce
```

In [ ]:
# Transformations — these DO NOTHING yet, just build the plan
rdd = sc.parallelize([1, 2, 3, 4, 5, 6])

doubled = rdd.map(lambda x: x * 2)         # LAZY — no computation
evens   = doubled.filter(lambda x: x % 2 == 0)  # LAZY — no computation

print('Nothing has run yet. doubled and evens are just plans.')

In [ ]:
# Action — NOW the computation actually happens
result = evens.collect()    # collect brings results to the driver
print('Result after action:', result)

Until `.collect()` is called, no actual computation runs on the cluster. Spark only **plans** the work.

## 4 — Common Transformations

### `map` — apply function to each element

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
squared = rdd.map(lambda x: x ** 2)
print('Squared:', squared.collect())
# [1, 4, 9, 16, 25]

### `filter` — keep elements matching a condition

In [ ]:
rdd = sc.parallelize(range(1, 11))
evens = rdd.filter(lambda x: x % 2 == 0)
print('Evens:', evens.collect())
# [2, 4, 6, 8, 10]

### `flatMap` — like map but flattens nested results

In [ ]:
sentences = sc.parallelize(['hello world', 'spark is great', 'rdd is the foundation'])

# map: each sentence → list of words (nested)
mapped = sentences.map(lambda s: s.split())
print('map (nested):', mapped.collect())

# flatMap: each sentence → words, flattened
flat = sentences.flatMap(lambda s: s.split())
print('flatMap (flat):', flat.collect())

Notice the difference:
```
map:     [['hello', 'world'], ['spark', 'is', 'great'], ...]
flatMap: ['hello', 'world', 'spark', 'is', 'great', ...]
```

`flatMap` is essential for word counts and tokenisation.

### `distinct` — remove duplicates

In [ ]:
rdd = sc.parallelize([1, 2, 2, 3, 3, 3, 4, 4, 4, 4])
unique = rdd.distinct()
print('Distinct:', sorted(unique.collect()))
# [1, 2, 3, 4]

## 5 — Common Actions

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print('count:    ', rdd.count())              # total elements
print('first:    ', rdd.first())              # first element
print('take(3):  ', rdd.take(3))              # first 3 elements
print('sum:      ', rdd.sum())                 # sum of all
print('mean:     ', rdd.mean())                # mean
print('max:      ', rdd.max())
print('reduce:   ', rdd.reduce(lambda a, b: a + b))  # custom reduction

### ⚠ `.collect()` — Use With Caution

`.collect()` brings **all data to the driver**. For big datasets → out-of-memory crash. Use `.take(n)` for inspection instead.

In [ ]:
# Small RDD — collect is fine
small = sc.parallelize([1, 2, 3])
print(small.collect())

# Big RDD — use take(n) instead
big = sc.parallelize(range(1_000_000))
print('Just inspect first 5:', big.take(5))

## 6 — Key/Value Pair RDDs

Many real-world RDDs are pairs `(key, value)`. Spark provides special operations like `reduceByKey` and `groupByKey`.

### `reduceByKey` — aggregate per key (most efficient)

In [ ]:
pairs = sc.parallelize([('a', 1), ('b', 2), ('a', 3), ('b', 4), ('a', 5)])

# Sum values per key
totals = pairs.reduceByKey(lambda a, b: a + b)
print('reduceByKey:', sorted(totals.collect()))
# [('a', 9), ('b', 6)]

### `groupByKey` — gather all values per key (less efficient)

In [ ]:
pairs = sc.parallelize([('a', 1), ('b', 2), ('a', 3), ('b', 4), ('a', 5)])

grouped = pairs.groupByKey()
result = [(k, list(v)) for k, v in grouped.collect()]
print('groupByKey:', sorted(result))
# [('a', [1, 3, 5]), ('b', [2, 4])]

**`reduceByKey` is preferred over `groupByKey`** — it pre-aggregates per partition before shuffling, so less data moves across the network.

### `sortByKey` — sort by key

In [ ]:
pairs = sc.parallelize([('b', 2), ('a', 1), ('c', 3), ('a', 4)])
print('Sorted by key:', pairs.sortByKey().collect())
# [('a', 1), ('a', 4), ('b', 2), ('c', 3)]

## 7 — Word Count (The Canonical RDD Example)

Combines `flatMap`, `map`, `reduceByKey`, and an action.

In [ ]:
text = sc.parallelize([
    'the quick brown fox',
    'the lazy dog',
    'the fox jumps over the dog'
])

word_counts = (text
    .flatMap(lambda line: line.split())          # split into words (flat)
    .map(lambda word: (word, 1))                  # pair each word with 1
    .reduceByKey(lambda a, b: a + b))             # sum per word

for word, count in sorted(word_counts.collect()):
    print(f'{word:12} {count}')

This is the **"Hello World" of distributed computing**. Same pattern handles terabytes of text — just point at a different data source.

## 8 — Caching / Persistence

If you use an RDD multiple times, **cache it**. Otherwise Spark recomputes it each time.

In [ ]:
rdd = sc.parallelize(range(1, 1_000_000))

expensive = rdd.map(lambda x: x ** 2).filter(lambda x: x % 3 == 0)

# Without caching — both actions trigger full recomputation
import time

t0 = time.time()
print('count:', expensive.count())   # computes the chain
t1 = time.time()
print('first:', expensive.first())   # computes AGAIN!
t2 = time.time()
print(f'\nWithout cache: count {t1-t0:.2f}s, first {t2-t1:.2f}s')

In [ ]:
# WITH caching
expensive_cached = rdd.map(lambda x: x ** 2).filter(lambda x: x % 3 == 0).cache()

t0 = time.time()
print('count:', expensive_cached.count())   # computes and caches
t1 = time.time()
print('first:', expensive_cached.first())   # uses cache — much faster
t2 = time.time()
print(f'\nWith cache: count {t1-t0:.2f}s, first {t2-t1:.2f}s')

expensive_cached.unpersist()   # free the cache when done

Caching makes repeated actions much faster — essential for iterative algorithms (ML training, graph processing).

## 9 — Lineage (Fault Tolerance)

Each RDD remembers HOW it was created. This is the **lineage** — used to recompute lost partitions on failure.

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
step1 = rdd.map(lambda x: x * 2)
step2 = step1.filter(lambda x: x > 4)
step3 = step2.map(lambda x: x + 100)

# toDebugString shows the lineage
print(step3.toDebugString().decode())

The lineage shows the RECIPE for `step3`. If a partition is lost during execution, Spark replays this chain to recompute it — that's the "Resilient" in RDD.

No data backups needed. The recipe IS the resilience.

## 10 — Repartitioning

Control parallelism by changing partition count.

In [ ]:
rdd = sc.parallelize(range(100), numSlices=4)
print('Original partitions:', rdd.getNumPartitions())

# Increase or decrease (full shuffle — expensive)
more = rdd.repartition(10)
print('After repartition:', more.getNumPartitions())

# Decrease only (cheaper — no full shuffle)
fewer = rdd.coalesce(2)
print('After coalesce:', fewer.getNumPartitions())

Use `repartition` to redistribute (increase or decrease); use `coalesce` to decrease cheaply.

Rule of thumb: **2-4× number of cores** in the cluster.

## 11 — Set-Like Operations

`union`, `intersection`, `subtract`.

In [ ]:
a = sc.parallelize([1, 2, 3, 4])
b = sc.parallelize([3, 4, 5, 6])

print('union:        ', sorted(a.union(b).collect()))         # all elements (with duplicates)
print('intersection: ', sorted(a.intersection(b).collect()))  # common elements
print('subtract:     ', sorted(a.subtract(b).collect()))      # in a but not b
print('union+distinct:', sorted(a.union(b).distinct().collect()))

## 12 — Sampling

In [ ]:
rdd = sc.parallelize(range(100))

# Sample 10% with replacement
sample = rdd.sample(withReplacement=False, fraction=0.1, seed=42)
print('Sample (~10):', sorted(sample.collect()))

Useful for exploring big datasets without processing everything.

## 13 — RDD → DataFrame

RDDs are low-level. Modern Spark code uses DataFrames. Convert easily:

In [ ]:
# RDD of tuples → DataFrame
rdd = sc.parallelize([('Alice', 29), ('Bob', 35), ('Carol', 30)])
df = rdd.toDF(['name', 'age'])
df.show()

# DataFrame → RDD
back_to_rdd = df.rdd
print(back_to_rdd.collect())

## 14 — Cleanup

In [ ]:
spark.stop()
print('Spark stopped.')

## Summary — RDD Cheat Sheet

```
Creation:
   sc.parallelize(list)           → RDD from Python list
   sc.textFile(path)               → RDD from file (one element per line)

Transformations (LAZY — return new RDD):
   .map(f)                          → apply f to each element
   .filter(f)                       → keep elements where f is True
   .flatMap(f)                      → map + flatten
   .distinct()                      → remove duplicates
   .union(rdd2)                     → combine
   .intersection(rdd2)              → common elements
   .subtract(rdd2)                  → in this, not other
   .sample(replace, fraction, seed) → random sample
   .reduceByKey(f)                  → aggregate by key (preferred)
   .groupByKey()                    → group all values by key
   .sortByKey()                     → sort by key
   .repartition(n)                  → change partition count (shuffle)
   .coalesce(n)                     → decrease partitions cheaply

Actions (EAGER — return value or write):
   .count()                         → number of elements
   .first()                         → first element
   .take(n)                         → first n elements
   .collect()                       → ALL elements (careful with big data)
   .sum() .mean() .max() .min()     → aggregations
   .reduce(f)                       → custom reduction
   .saveAsTextFile(path)            → write to storage

Caching:
   .cache()                          → in-memory
   .persist(StorageLevel)            → configurable
   .unpersist()                      → release
```

> RDDs are the foundation of Spark. Modern code prefers DataFrames, but understanding RDDs is essential for debugging, performance tuning, and interviews.